# Imports

In [0]:
from pyspark.sql.functions import trim, when, length, lit, col, row_number, lower as lower_spark, concat_ws, coalesce, current_timestamp, sha2
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.events"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.races"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

# Metodos

In [0]:
def clean_string(column_name: str):
    value = trim(col(column_name))

    return (
        when(length(value) == 0, lit(None).cast("string"))
         .otherwise(value)
    )

# Races

In [0]:
bronze_df = spark.table(BRONZE_TABLE)

# ---------------------------------------------------------
# 1. Deduplicación
# ---------------------------------------------------------
# Si el mismo events.parquet fue reingestado,
# conservamos la versión más reciente de cada carrera.
window_latest = (
    Window
    .partitionBy("season", "RoundNumber")
    .orderBy(
        col("_source_file_modification_time").desc_nulls_last(),
        col("_ingested_at").desc_nulls_last()
    )
)

latest_df = (
    bronze_df
    .withColumn("_row_number", row_number().over(window_latest))
    .filter(col("_row_number") == 1)
    .drop("_row_number")
)

# ---------------------------------------------------------
# 2. No permitir rescued data en Silver
# ---------------------------------------------------------
rescued_count = (
    latest_df
    .filter(col("_rescued_data").isNotNull())
    .limit(1)
    .count()
)

if rescued_count > 0:
    raise ValueError(
        f"{BRONZE_TABLE} contiene registros con _rescued_data. "
        "Silver no será actualizada."
    )

# ---------------------------------------------------------
# 3. Normalización
# ---------------------------------------------------------
races_df = latest_df.select(
    col("season").cast("int").alias("season"),
    col("RoundNumber").cast("int").alias("round"),

    clean_string("Country").alias("country"),
    clean_string("Location").alias("location"),

    clean_string("OfficialEventName").alias("official_event_name"),

    col("EventDate")
        .cast("date")
        .alias("event_date"),

    clean_string("EventName").alias("event_name"),

    lower_spark(
        clean_string("EventFormat")
    ).alias("event_format"),

    clean_string("Session1").alias("session_1_name"),
    col("Session1DateUtc")
        .cast("timestamp_ntz")
        .alias("session_1_start_utc"),

    clean_string("Session2").alias("session_2_name"),
    col("Session2DateUtc")
        .cast("timestamp_ntz")
        .alias("session_2_start_utc"),

    clean_string("Session3").alias("session_3_name"),
    col("Session3DateUtc")
        .cast("timestamp_ntz")
        .alias("session_3_start_utc"),

    clean_string("Session4").alias("session_4_name"),
    col("Session4DateUtc")
        .cast("timestamp_ntz")
        .alias("session_4_start_utc"),

    clean_string("Session5").alias("session_5_name"),
    col("Session5DateUtc")
        .cast("timestamp_ntz")
        .alias("session_5_start_utc"),

    col("F1ApiSupport")
        .cast("boolean")
        .alias("f1_api_support"),

    col("_source_file").alias("source_file"),

    col("_source_file_modification_time")
        .alias("source_modified_at"),

    col("_ingested_at")
        .alias("bronze_ingested_at")
)

# ---------------------------------------------------------
# 4. Hash del contenido de negocio
# ---------------------------------------------------------
business_columns = [
    "season",
    "round",
    "country",
    "location",
    "official_event_name",
    "event_date",
    "event_name",
    "event_format",
    "session_1_name",
    "session_1_start_utc",
    "session_2_name",
    "session_2_start_utc",
    "session_3_name",
    "session_3_start_utc",
    "session_4_name",
    "session_4_start_utc",
    "session_5_name",
    "session_5_start_utc",
    "f1_api_support"
]

hash_expression = concat_ws(
    "||",
    *[
        coalesce(
            col(column).cast("string"),
            lit("<NULL>")
        )
        for column in business_columns
    ]
)

races_df = (
    races_df
    .withColumn(
        "record_hash",
        sha2(hash_expression, 256)
    )
    .withColumn(
        "silver_updated_at",
        current_timestamp()
    )
)

races_df.display()



## Validaciones - Races

In [0]:
validation = (
    races_df
    .agg(
        sum_spark(
            when(col("season").isNull(), 1).otherwise(0)
        ).alias("null_season"),

        sum_spark(
            when(
                col("round").isNull() |
                (col("round") <= 0),
                1
            ).otherwise(0)
        ).alias("invalid_round"),

        sum_spark(
            when(col("country").isNull(), 1).otherwise(0)
        ).alias("null_country"),

        sum_spark(
            when(col("location").isNull(), 1).otherwise(0)
        ).alias("null_location"),

        sum_spark(
            when(col("event_name").isNull(), 1).otherwise(0)
        ).alias("null_event_name"),

        sum_spark(
            when(col("event_date").isNull(), 1).otherwise(0)
        ).alias("null_event_date"),

        sum_spark(
            when(col("event_format").isNull(), 1).otherwise(0)
        ).alias("null_event_format"),

        sum_spark(
            when(col("season") < 1950, 1).otherwise(0)
        ).alias("invalid_season")
    )
    .first()
    .asDict()
)

print(validation)

errors = {
    rule: value or 0
    for rule, value in validation.items()
    if (value or 0) > 0
}

# La clave (season, round) debe ser única
duplicate_count = (
    races_df
    .groupBy("season", "round")
    .count()
    .filter(col("count") > 1)
    .count()
)

if duplicate_count > 0:
    errors["duplicate_season_round"] = duplicate_count

if errors:
    raise ValueError(
        f"Silver races validation failed: {errors}"
    )

print(
    f"Validation OK: {races_df.count()} races ready for Silver."
)

## Insert - Races

In [0]:
if spark.catalog.tableExists(SILVER_TABLE):
    target = DeltaTable.forName(spark, SILVER_TABLE)

    (
        target.alias("target")
        .merge(
            races_df.alias("source"),
            """
            target.season = source.season
            AND target.round = source.round
            """
        )
        .whenMatchedUpdateAll(
            condition="""
                target.record_hash <> source.record_hash
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Merged data into {SILVER_TABLE}")

else:
    (
        races_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(SILVER_TABLE)
    )

    print(f"Created {SILVER_TABLE}")